# 📊 Poll Results Visualizer — Exploratory Data Analysis

**Project:** Poll Results Visualizer  
**Author:** Your Name  
**Date:** 2024  
**Goal:** Analyse a 1,000-respondent product preference survey and extract actionable insights.

---

## Workflow
```
Data Generation → Loading → Cleaning → EDA → Analysis → Visualization → Insights
```

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('✅ Libraries loaded successfully')

## 2. Generate & Load Data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.generate_data import generate_poll_data
from src.analysis import load_and_clean

# Generate synthetic data
raw = generate_poll_data(n=1000)
raw.to_csv('../data/poll_data.csv', index=False)

# Load and clean
df = load_and_clean('../data/poll_data.csv')
print(f'Shape: {df.shape}')
df.head()

## 3. Data Overview

In [ ]:
print('=== DATA TYPES ===')
print(df.dtypes)
print('\n=== NULL VALUES ===')
print(df.isnull().sum())
print('\n=== BASIC STATS (numeric) ===')
df.describe()

In [ ]:
print('=== CATEGORICAL DISTRIBUTIONS ===')
for col in ['region', 'age_group', 'gender', 'education', 'preferred_product',
            'satisfaction', 'nps_category', 'would_buy_again']:
    print(f'\n{col}:')
    print(df[col].value_counts())

## 4. Overall Vote Analysis

In [ ]:
from src.analysis import overall_product_votes
votes = overall_product_votes(df)
print('=== VOTE SHARE BY PRODUCT ===')
print(votes.to_string(index=False))

In [ ]:
PALETTE = ['#2563EB','#16A34A','#DC2626','#D97706','#7C3AED']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar
axes[0].barh(votes['product'][::-1], votes['percentage'][::-1],
             color=PALETTE[::-1], edgecolor='white', height=0.55)
for i, (_, row) in enumerate(votes[::-1].iterrows()):
    axes[0].text(row['percentage'] + 0.3, i, f"{row['percentage']}%",
                 va='center', fontweight='bold')
axes[0].set_title('Vote Share (Bar)', fontweight='bold')
axes[0].set_xlabel('Vote Share (%)')
axes[0].spines[['top','right']].set_visible(False)

# Pie
axes[1].pie(votes['votes'], labels=votes['product'], autopct='%1.1f%%',
            colors=PALETTE, startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Vote Distribution (Pie)', fontweight='bold')

plt.suptitle('Overall Product Preference', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/eda_01_overall.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Regional Analysis

In [ ]:
from src.analysis import region_wise_analysis
pivot, pct = region_wise_analysis(df)

print('=== RAW COUNTS ===')
print(pivot)
print('\n=== PERCENTAGE WITHIN REGION ===')
print(pct)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
sns.heatmap(pct, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, ax=axes[0], cbar_kws={'label': 'Vote Share (%)'})
axes[0].set_title('Region × Product Heatmap (%)', fontweight='bold')

# Stacked
pct.plot(kind='bar', stacked=True, ax=axes[1],
         color=PALETTE, edgecolor='white', linewidth=0.5)
axes[1].set_title('Stacked Regional Preference', fontweight='bold')
axes[1].set_ylabel('Vote Share (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Product', bbox_to_anchor=(1.01, 1), fontsize=9)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/eda_02_regional.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Demographic Analysis

In [ ]:
from src.analysis import age_wise_analysis, gender_wise_analysis

_, age_pct    = age_wise_analysis(df)
_, gender_pct = gender_wise_analysis(df)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

age_pct.plot(kind='bar', ax=axes[0], color=PALETTE, edgecolor='white', width=0.75)
axes[0].set_title('Age Group Preference', fontweight='bold')
axes[0].set_ylabel('Vote Share (%)')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Product', fontsize=9)
axes[0].spines[['top','right']].set_visible(False)

gender_pct.plot(kind='bar', ax=axes[1], color=PALETTE, edgecolor='white', width=0.65)
axes[1].set_title('Gender-wise Preference', fontweight='bold')
axes[1].set_ylabel('Vote Share (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Product', fontsize=9)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/eda_03_demographic.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Satisfaction & NPS

In [ ]:
from src.analysis import satisfaction_analysis, nps_analysis

sat          = satisfaction_analysis(df)
nps, score   = nps_analysis(df)

print(f'Satisfaction breakdown:\n{sat}\n')
print(f'NPS breakdown:\n{nps}\n')
print(f'📊 NPS Score: {score}')

## 8. Trend Analysis

In [ ]:
from src.analysis import monthly_trend

trend = monthly_trend(df)
print('Monthly vote counts per product:')
print(trend)

fig, ax = plt.subplots(figsize=(13, 5))
for i, col in enumerate(trend.columns):
    ax.plot(trend.index, trend[col], marker='o', linewidth=2.2,
            color=PALETTE[i % len(PALETTE)], label=col)
ax.set_title('Monthly Voting Trend by Product', fontweight='bold', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Votes')
ax.legend(title='Product', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../outputs/eda_04_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Correlation Analysis

In [ ]:
# Encode categoricals for correlation
from sklearn.preprocessing import LabelEncoder

df_enc = df.copy()
for col in ['region','age_group','gender','education','preferred_product',
            'satisfaction','nps_category','would_buy_again']:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

corr = df_enc[['preferred_product','rating','region','age_group',
               'gender','satisfaction','nps_category']].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_05_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Final Insights Summary

In [ ]:
from src.analysis import generate_insights

print('=' * 55)
print('   KEY INSIGHTS — POLL RESULTS VISUALIZER')
print('=' * 55)
for insight in generate_insights(df):
    print(f'  {insight}')
print('=' * 55)

## ✅ Notebook Complete

All charts saved to `outputs/`. Run `python main.py` for the full CLI pipeline or `streamlit run app.py` for the dashboard.